This implementation extends the Grouped Query Attention (GQA) language model with looped transformer depth. The same transformer-layer stack is applied multiple times, increasing effective depth without adding another set of transformer-block parameters.


In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import torch.nn as nn
import time
from transformers import AutoTokenizer
import numpy as np
import math
from llm_module import (
    Model, GroupedQueryAttention, LMDataset, generate_sample, train, calculate_loader_loss,
    calculate_perplexity, generate, generate_text_stream_cache
)
import json
import os

Similar to my other implementation, I will use the RedPajama dataset and the gpt-2 tokenizer to keep the vocab size limited due to hardware limitations.

In [2]:
tokenizer = AutoTokenizer.from_pretrained('gpt2')

In [3]:
# Reusing local data generated by original notebook
train_path = 'train.bin'
val_path = 'val.bin'
test_path = 'test.bin'

CONTEXT_LENGTH = 512

train_dataset = LMDataset(train_path, CONTEXT_LENGTH)
val_dataset = LMDataset(val_path, CONTEXT_LENGTH)
test_dataset = LMDataset(test_path, CONTEXT_LENGTH)

In [4]:
# Test dataset
print(train_dataset.tokens[:10])

[15045 11942   860  4790    12 26660    12 34215   198 26953]


In [5]:
BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

In [6]:
# Test dataloader
example = next(iter(train_loader))
print(example)

[tensor([[15045, 11942,   860,  ...,    11,   347,  5488],
        [   11,  1395,    13,  ...,   326,   262,  1074],
        [  290, 27686,   389,  ...,   319,  3592,    13],
        ...,
        [  898,  7883,   290,  ...,   468,  2722,  3068],
        [  422,  1811, 12527,  ...,  8429,  4129,    13],
        [  198,   464,   734,  ..., 17514,  8171,    11]]), tensor([[11942,   860,  4790,  ...,   347,  5488,    11],
        [ 1395,    13,   406,  ...,   262,  1074,   290],
        [27686,   389,   564,  ...,  3592,    13,  2773],
        ...,
        [ 7883,   290,   703,  ...,  2722,  3068,   422],
        [ 1811, 12527,   329,  ...,  4129,    13,   198],
        [  464,   734, 48924,  ...,  8171,    11,   340]])]


## Prepare the model

This implementation uses Grouped Query Attention and repeatedly applies the same transformer-layer stack. Each loop shares weights, while each block application uses a separate KV-cache entry.


#### Initialize the model

In [7]:
class LoopedModel(Model):
    def __init__(self, cfg, xsa=False):
        super().__init__(cfg, xsa=xsa)
        self.n_loops = cfg['n_loops']
        self.n_shared_layers = cfg['n_layers']

        # Cache entries belong to block applications, not parameter sets.
        self.cfg = dict(cfg)
        self.cfg['n_layers'] = self.n_shared_layers * self.n_loops

    def forward(self, x, cache=None):
        x = self.tok_emb(x)
        num_tokens = x.shape[1]

        start = 0
        if cache:
            start = self.current_pos
            self.current_pos += num_tokens

        # Main change for looped-transformer
        for loop_idx in range(self.n_loops):
            for layer_idx, block in enumerate(self.trf_blocks):
                cache_idx = loop_idx * self.n_shared_layers + layer_idx
                block_cache = cache.get(cache_idx) if cache else None
                x, next_cache = block(
                    x, self.cos, self.sin, start, block_cache
                )
                if cache is not None:
                    cache.update(cache_idx, next_cache)

        x = self.norm(x)
        return self.out_head(x)


MODEL_CONFIG = {
    'vocab_size': 50257,
    'context_length': CONTEXT_LENGTH,
    'n_layers': 13,
    'n_loops': 2,
    'n_heads': 8,
    'n_kv_groups': 4,
    'emb_dim': 768,
    'hidden_dim': 2048,
    'dtype': torch.bfloat16
}

model = LoopedModel(MODEL_CONFIG, xsa=False)

In [8]:
# Using mps, modify for cuda
device = torch.device('mps' if torch.mps.is_available() else 'cpu')

print('Device:', device)
model.to(device)

Device: mps


LoopedModel(
  (tok_emb): Embedding(50257, 768)
  (trf_blocks): ModuleList(
    (0-12): 13 x Transformer(
      (norm1): RMSNorm()
      (norm2): RMSNorm()
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=384, bias=False)
        (W_value): Linear(in_features=768, out_features=384, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
        (out_proj): Linear(in_features=768, out_features=768, bias=False)
      )
      (mlp): MLP(
        (fc1): Linear(in_features=768, out_features=2048, bias=True)
        (fc2): Linear(in_features=768, out_features=2048, bias=True)
        (fc3): Linear(in_features=2048, out_features=768, bias=True)
      )
    )
  )
  (norm): RMSNorm()
  (out_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [9]:
parameters = sum(p.numel() for p in model.parameters())
print(f'Parameters: {parameters:,}')

Parameters: 123,028,672


In [10]:
# Test model forward passes and generate function
generate_sample(
    model=model,
    tokenizer=tokenizer,
    device=device,
    context='Dedication will always pay',
    context_size=MODEL_CONFIG['context_length']
)

Dedication will always pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay pay


### Training

In [11]:
model = torch.compile(model, dynamic=True)

In [ ]:
torch.manual_seed(123)

num_epochs = 1
initial_lr = 1e-5
min_lr = 4e-5

optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, fused=True)

warmup_steps = int((len(train_loader) * num_epochs) * 0.05)

# Improve training time on consumer laptop
GRAD_ACCUM_STEPS = 2

start_training_time = time.time()
train_losses, val_losses, train_perplexities, val_perplexities, tokens_seen = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=num_epochs,
    optimizer=optimizer,
    eval_freq=10000,
    eval_batches=10,
    message_freq=20000,
    tokenizer=tokenizer,
    device=device,
    warmup_steps=warmup_steps,
    initial_lr=initial_lr,
    min_lr=min_lr,
    model_name='looped-transformer.pth'
)
end_training_time = time.time()

In [ ]:
torch.save({
    'step': 170090,
    'model': model.state_dict(),
    'optimizer': optimizer.state_dict(),
}, 'looped-transformer.pth')

In [ ]:
print(f'Elapsed time: {(end_training_time-start_training_time)/60:.2f} minutes')

In [ ]:
steps = np.arange(0, len(train_losses) * 10000, 10000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(steps, train_losses, label='Train')
ax1.plot(steps, val_losses, label='Validation')
ax1.set_xlabel('Steps')
ax1.set_ylabel('Loss')
ax1.set_title('Loss over training steps')
ax1.legend()

ax2.plot(steps, train_perplexities, label='Train')
ax2.plot(steps, val_perplexities, label='Validation')
ax2.set_xlabel('Steps')
ax2.set_ylabel('Perplexity')
ax2.set_title('Perplexity over training steps')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
model.eval()

with torch.no_grad():
    test_loss = calculate_loader_loss(
        model=model,
        loader=test_loader,
        eval_batches=10, 
        device=device
    )

print(f'Test loss: {test_loss:.3f}')

test_perplexity = calculate_perplexity(test_loss)
print(f'Test perplexity: {test_perplexity:.2f}')

In [ ]:
text1 = 'Dedication will always pay'
text2 = 'Large language models learn by'

text1_tokens = tokenizer.encode(text1, return_tensors='pt').to(device)
text2_tokens = tokenizer.encode(text2, return_tensors='pt').to(device)

print('Generated from example 1:\n')
generated_tokens_1 = generate(
    model=model,
    tokenizer=tokenizer,
    prompt=text1,
    device=device,
    verbose=True,
    max_new_tokens=50,
    temperature=1.2, 
    top_k=20
)
print('\n\n\nGenerated from example 2:\n')
generated_tokens_2 = generate(
    model=model,
    tokenizer=tokenizer,
    prompt=text2,
    device=device,
    verbose=True,
    max_new_tokens=50,
    temperature=1.2, 
    top_k=20
)

In [ ]:
metrics = {
    "model": "looped-transformer",
    "train_losses": train_losses,
    "val_losses": val_losses,
    "train_perplexities": [p.item() if hasattr(p, "item") else p for p in train_perplexities],
    "val_perplexities": [p.item() if hasattr(p, "item") else p for p in val_perplexities],
}

metrics_path = "looped-transformer-metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved metrics to {os.path.abspath(metrics_path)}")